In [ ]:
# Example 1: DictPath basic traversal of a nested dict via __truediv__.
# pathable 0.6+: DictPath is an alias of LookupPath — construct via from_lookup();
# content() was renamed read_value().
from pathable import DictPath

data = {
    "servers": {
        "primary": {"host": "db-1.example.com", "port": 5432},
        "replica": {"host": "db-2.example.com", "port": 5433},
    }
}

root = DictPath.from_lookup(data)
primary_host = root / "servers" / "primary" / "host"

print(f"parts = {primary_host.parts}")
print(f"read_value() = {primary_host.read_value()}")
assert primary_host.read_value() == "db-1.example.com"

In [ ]:
# Example 2: DictPath iteration over keys at a level.
from pathable import DictPath

data = {
    "users": {
        "alice": {"role": "admin"},
        "bob": {"role": "viewer"},
        "carol": {"role": "editor"},
    }
}

users_path = DictPath.from_lookup(data) / "users"

child_keys = sorted(child.parts[-1] for child in users_path)
print(f"child keys at users/ = {child_keys}")
assert child_keys == ["alice", "bob", "carol"]
assert sorted(users_path.keys()) == ["alice", "bob", "carol"]

alice_role = (users_path / "alice" / "role").read_value()
print(f"alice.role = {alice_role}")
assert alice_role == "admin"

In [ ]:
# Example 3: DictPath with list-valued nodes (mixed dict/list traversal via integer indices).
from pathable import DictPath

data = {
    "team": {
        "members": [
            {"name": "Ada", "skills": ["math", "engines"]},
            {"name": "Grace", "skills": ["compilers", "navy"]},
        ]
    }
}

root = DictPath.from_lookup(data)
first_member_name = (root / "team" / "members" / 0 / "name").read_value()
first_skill = (root / "team" / "members" / 0 / "skills" / 0).read_value()

print(f"first member name = {first_member_name}")
print(f"first member skill[0] = {first_skill}")
assert first_member_name == "Ada"
assert first_skill == "math"

In [ ]:
# Example 4: AccessorPath / LookupAccessor via a Mapping-backed lookup.
from pathable.accessors import LookupAccessor
from pathable.paths import AccessorPath

lookup = {
    "config": {
        "feature_flags": {
            "beta_ui": True,
            "dark_mode": False,
        }
    }
}

accessor = LookupAccessor(lookup)
root = AccessorPath(accessor)

beta = (root / "config" / "feature_flags" / "beta_ui").read_value()
dark = (root / "config" / "feature_flags" / "dark_mode").read_value()
print(f"beta_ui = {beta}, dark_mode = {dark}")
assert beta is True
assert dark is False

In [ ]:
# Example 5: Path equality, repr, and parts semantics across operations.
from pathable import DictPath

data = {"a": {"b": {"c": 42}}}

root = DictPath.from_lookup(data)
p1 = root / "a" / "b" / "c"
p2 = (root / "a") / "b" / "c"

print(f"p1.parts = {p1.parts}")
print(f"p2.parts = {p2.parts}")
print(f"p1.read_value() = {p1.read_value()}")

# Same logical path constructed two ways should yield the same parts and content.
assert tuple(p1.parts) == tuple(p2.parts) == ("a", "b", "c")
assert p1 == p2  # same address, same accessor binding
assert p1.read_value() == p2.read_value() == 42

# Path is iterable as a sequence of child paths over the value at p1's parent.
b_path = root / "a" / "b"
child_keys = sorted(child.parts[-1] for child in b_path)
assert child_keys == ["c"]
print("All pathable assertions passed.")